#1~4で最も高いスコアを記録したデータ・モデルを用いて、さらに改良を行う。

In [45]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [3]:
#データフレームの読み込み
'''
以降はスコアが最高であったdf3を用いて分析を行う。
sc_x,df_yはdf3を基に作成する。
ただし、必要があればdf1, df2も利用する。
'''
df3 = pd.read_csv('datafiles/df3_lowered_VIF.csv')

sc_x = df3.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df3['SalePrice'])

In [4]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [5]:
#リッジ回帰のalphaを90~1000で最適化（以前は１～100までの自然数で実験しただけであった）
best_ridgescore = 0
best_alpha = 0

for i in range(90,1000):
    ridgeModel = Ridge(random_state = 0, alpha = i)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = i
print(f'最適な正則化項＝{best_alpha}　最高スコア＝{best_ridgescore}')

最適な正則化項＝478　最高スコア＝0.8205844141029276


In [6]:
#リッジ回帰のalphaを478付近でより細かく最適化（以前は１～100までの自然数で実験しただけであった）
best_ridgescore = 0
best_alpha = 0

for i in range(47700,47900):
    num = i/100
    ridgeModel = Ridge(random_state = 0, alpha = num)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = num
print(f'最適な正則化項＝{best_alpha}　最高スコア＝{best_ridgescore}')


最適な正則化項＝478.14　最高スコア＝0.8205844152693619


In [7]:
#4_analysis_2においてVIFを行の削除により下げたスコアの最高値   0.7971109145093968よりもスコアが向上した。

In [8]:
#引き続き、特徴量の最適化を行う。
model5 = Ridge(random_state = 0, alpha = best_alpha)
model5.fit(sc_x, df_y)
coef_df = pd.DataFrame({
    'features':sc_x.columns,
    'Coefficient': model5.coef_
})
coef_df.sort_values('Coefficient', ascending=False)

,features,Coefficient
2,OverallQual,10957.831681
8,1stFlrSF,9102.571118
16,TotRmsAbvGrd,7851.661035
195,Neighborhood_NoRidge,6941.700492
196,Neighborhood_NridgHt,6872.611172
...,...,...
75,BldgType_TwnhsE,-3760.381477
127,KitchenQual_Gd,-4433.561103
48,BsmtQual_Gd,-4760.329032
97,PoolQC_Gd,-4799.248606


In [36]:
'''
今回のデータセットでは各特徴量の意味が明確であるから、ドメイン知識に基づいて仮説検証を行う。
以下が検証すべき仮説である。

・都会の一戸建ては価格が高い。
・好立地で敷地面積が大きいと価格は高くなりやすい。
・都会の中でも住宅街のような閑静なエリアは価格が高い。
・寒い地域で暖房設備に欠陥があったり、断熱性の弱い外壁素材であったりすると価格が低い。
・プール、地下室など生活のために必須ではない要素がある家には富裕層が住んでいる可能性が高いため価格も高い。
・特に立地・住宅の種類は、他の様々な特徴量と関連性が強そうである。

ゆえに、以下の順番で検証を行う。
1.Neighborhood（立地）に関連する各ダミー変数列と、各特徴量の交互作用特徴量を作成
2.MSSubClass（住宅の種類）と、各特徴量の交互作用特徴量を作成
3.BsmtQual_NA（地下室の有無）、WoodDeckSF、OpenPorchSF、EnclosedPorch、3SsnPorch、ScreenPorch、
  PoolArea、MiscFeature（家に付随する必須ではない機能）は関連性が高い可能性があるので、
  PCAにより特徴量を統合することや、クラスタリングでの分類を目指す。

検証のために必要な特徴量を削除してしまっていたので、以下ではdf1（特徴量削除前のデータフレーム）を用いる。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df3['SalePrice'])


In [37]:
result = cross_validate(model5, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.8192748628393769


In [38]:
#元の列をリスト化しておく
original_cols = list(sc_x.columns)

#1の仮説検証

In [ ]:
#sc_x1を１の仮説検証のためのデータフレームとする。
sc_x1 = sc_x.copy()

Neighborhood_cols = []
for c in original_cols:
    if 'Neighborhood' in c:
        Neighborhood_cols.append(c)

not_neighborhood_cols = list(set(original_cols) - set(Neighborhood_cols))

#Neighborhood_colsそれぞれに対して多項式特徴量を追加
new_cols = {}
for c1 in Neighborhood_cols:
    for c2 in not_neighborhood_cols:
        col_name = c1 + '_' + c2
        new_cols[col_name] = sc_x1[c1] * sc_x1[c2]
sc_x1 = pd.concat([sc_x1, pd.DataFrame(new_cols, index=sc_x1.index)], axis=1)

In [40]:
#作成したデータで学習
model5 = Ridge(alpha = best_alpha)
result = cross_validate(model5, sc_x1, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel5のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel5のスコア＝0.838013880006198


In [47]:
#1により、スコアは誤差程度向上した。ただし、余計な特徴量を非常に多く追加してしまった可能性がある。
#したがって、効果の低い特徴量の係数を０にできるラッソ回帰が有効である可能性がある。

#一旦alpha=478のラッソ回帰モデルで学習
model6 = Lasso(alpha = 478, random_state = 0)
result = cross_validate(model6, sc_x1, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel3のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel3のスコア＝0.84303517031005


In [49]:
#ゆえに、ラッソ回帰によりスコアが大幅に向上した。

#最適なalphaの探索
best_lassoscore = 0
best_alpha = 0
#alpha（Fの係数）を100,200,…,1000まで変化させて実験
for i in range(1,11):
    num = i*100
    lassoModel = Lasso(random_state = 0, alpha = num)
    all_result = cross_validate(lassoModel, sc_x1, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = num
print(f'最適な正則化項＝{best_alpha}　ラッソ回帰の最高スコア＝{best_lassoscore}')

c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.229e+09, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.094e+09, tolerance: 7.034e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.264e+09, toleranc

最適な正則化項＝300　ラッソ回帰の最高スコア＝0.844173171400673


In [50]:
#最適なalphaの探索
#alpha（Fの係数）を250,260,…,340,350まで変化させて実験
for i in range(25,36):
    num = i*10
    lassoModel = Lasso(random_state = 0, alpha = num)
    all_result = cross_validate(lassoModel, sc_x1, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = num
print(f'最適な正則化項＝{best_alpha}　ラッソ回帰の最高スコア＝{best_lassoscore}')

最適な正則化項＝330　ラッソ回帰の最高スコア＝0.8443803597436469


In [52]:
#最適なalphaの探索
#alpha（Fの係数）を325,326,…,335まで変化させて実験
for i in range(325,335):
    num = i
    lassoModel = Lasso(random_state = 0, alpha = num)
    all_result = cross_validate(lassoModel, sc_x1, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = num
print(f'最適な正則化項＝{best_alpha}　ラッソ回帰の最高スコア＝{best_lassoscore}')

model6 =  Lasso(random_state = 0, alpha = best_alpha)

最適な正則化項＝329　ラッソ回帰の最高スコア＝0.8443813089776464


In [ ]:
#1により、ラッソ回帰のスコアがリッジ回帰を上回った。
#以降はこのmodel6（正則化項＝329のラッソ回帰モデル）と、1により特徴量を追加したデータフレームsc_x1を利用する。

#2の仮説検証

In [53]:
#sc_x2を2の仮説検証のためのデータフレームとする。
sc_x2 = sc_x1.copy()

#多項式特徴量を追加
new_cols = {}
for c in original_cols:
    if c != 'MSSubClass':
        col_name = 'MSSubClass_' + c
        new_cols[col_name] = sc_x2['MSSubClass'] * sc_x2[c]
sc_x2 = pd.concat([sc_x2, pd.DataFrame(new_cols, index=sc_x2.index)], axis=1)

In [55]:
#作成したデータで学習
result = cross_validate(model6, sc_x2, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel6のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel6のスコア＝0.8409356168854382


In [ ]:
#2によりスコアが低下した。よって2での改変は棄却する。
#以降はmodel6とsc_x1を利用する。